In [ ]:
import numpy as np


import torch.nn as nn
import torch.optim as optim

from sklearn.metrics import balanced_accuracy_score, confusion_matrix


import matplotlib.pyplot as plt


from mil.CellsData import CellsData
from mil.CustomDataloader import CustomLoader

from mil.training_utils import model_run, set_seed, get_sub_dataset, stratified_cv_split
from mil import PROJECT_ROOT

from mil.models import (
    MIL_model,
    MLP_encoder,
    MaxAggergation,
    AttentionAggregation,
    GatedAttentionAggregation,
)

from bayes_opt import BayesianOptimization

train_set = CellsData(split="train")
validation_set = CellsData(split="val")


def model_test(
    encoder,
    encoder_settings,
    aggregator,
    aggregator_settings,
    learning_rate=0.001,
    decay=0.01,
    seeds=[37, 42],
    verbose=False,
):
    losses = []
    for seed in seeds:
        set_seed(seed)

        train_loader = CustomLoader(train_set, batchsize=20)
        validation_loader = CustomLoader(validation_set, batchsize=20)
        """
        table_row_to_number = MLP_encoder(
            n_hidden=n_hidden, hidden_size=hidden_size, output_size=1
        )
        max_aggregator = MaxAggergation(use_sigmoid=True)

        model = MIL_model(
            instance_encoder=table_row_to_number, bag_aggregator=max_aggregator
        )
        """

        encoder_model = encoder(**encoder_settings)
        aggregator_model = aggregator(**aggregator_settings)

        model = MIL_model(
            instance_encoder=encoder_model, bag_aggregator=aggregator_model
        )
        criterion = nn.BCELoss()
        optimizer = optim.Adam(model.parameters(), lr=learning_rate, weight_decay=decay)

        num_epochs = 30

        if verbose:
            fig, ax = plt.subplots()

        if verbose:
            ax = ax
        else:
            ax = None

        train_loss, valid_loss, best_epoch = model_run(
            model=model,
            train_loader=train_loader,
            validation_loader=validation_loader,
            criterion=criterion,
            optimizer=optimizer,
            num_epochs=num_epochs,
            save_path_prefix=str(PROJECT_ROOT / "data/max_aggregation_models/epoch_"),
            ax=ax,
            plot_title="Training of MIL Classifier with Max Aggregation",
            verbose=verbose,
        )

        min_loss = valid_loss[best_epoch]
        losses.append(min_loss)

    min_loss = np.min(losses)
    return min_loss


def test_max_model(n_hidden=3, hidden_size=15, learning_rate=0.001, decay=0.01):
    """Returns -minimal_loss (maximization is equivalent to loss minimization)

    Args:
        n_hidden (int, optional): _description_. Defaults to 3.
        hidden_size (int, optional): _description_. Defaults to 15.
        lr (float, optional): _description_. Defaults to 0.001.
        decay (float, optional): _description_. Defaults to 0.01.
    """
    n_hidden = int(np.floor(n_hidden))
    hidden_size = int(np.floor(hidden_size))
    settings = {
        "encoder": MLP_encoder,
        "encoder_settings": {
            "n_hidden": n_hidden,
            "hidden_size": hidden_size,
            "output_size": 1,
        },
        "aggregator": MaxAggergation,
        "aggregator_settings": {"use_sigmoid": True},
        "learning_rate": learning_rate,
        "decay": decay,
        "seeds": (0, 37, 42, 1999, 2023),
    }

    loss = model_test(**settings)
    return -loss


pbounds = {
    "hidden_size": (2, 50.5),
    "n_hidden": (1, 3.5),
    "learning_rate": (10**-4, 10**-2),
    "decay": (0.0001, 0.1),
}

f_to_maximize = test_max_model


optimizer = BayesianOptimization(
    f=f_to_maximize,
    pbounds=pbounds,
    verbose=2,  # verbose = 1 prints only when a maximum is observed, verbose = 0 is silent
    random_state=1,
)
optimizer.maximize(
    init_points=10,
    n_iter=10,
)

|   iter    |  target   |   decay   | hidden... | learni... | n_hidden  |
-------------------------------------------------------------------------
| 1         | -0.3337   | 0.04176   | 36.94     | 0.0001011 | 1.756     |
| 2         | -0.3022   | 0.01476   | 6.478     | 0.001944  | 1.864     |
| 3         | -0.3014   | 0.03974   | 28.13     | 0.00425   | 2.713     |
| 4         | -0.3248   | 0.02052   | 44.59     | 0.0003711 | 2.676     |
| 5         | -0.2819   | 0.04179   | 29.1      | 0.00149   | 1.495     |
| 6         | -0.2961   | 0.08009   | 48.96     | 0.003203  | 2.731     |
| 7         | -0.3101   | 0.08765   | 45.39     | 0.0009419 | 1.098     |
| 8         | -0.3061   | 0.01707   | 44.59     | 0.001074  | 2.053     |
| 9         | -0.2872   | 0.09579   | 27.86     | 0.00695   | 1.789     |
| 10        | -0.3253   | 0.06868   | 42.48     | 0.0002811 | 2.875     |
| 11        | -0.3267   | 0.1       | 28.42     | 0.0001    | 1.0       |
| 12        | -0.294    | 0.04431   | 

In [77]:
def run_optimizer(f_to_maximize, pbounds, init_points = 10, n_iter = 10, verbose = 2, random_state = 1):
    
    optimizer = BayesianOptimization(
        f=f_to_maximize,
        pbounds=pbounds,
        verbose=verbose,  # verbose = 1 prints only when a maximum is observed, verbose = 0 is silent
        random_state=random_state,
    )
    optimizer.maximize(
        init_points=init_points,
        n_iter=n_iter,
    )
    
    best_idx = np.argmax([el['target'] for el in optimizer.res])
    best_config = optimizer.res[best_idx]

    settings = yaml.dump(pbounds)
    config_dict = ast.literal_eval(str(best_config))
    result = yaml.dump(config_dict)

    now = datetime.now() 
    time = now.strftime('%Y-%m-%d %H:%M:%S')

    doc_string = f'{time}\n\nf: {f_to_maximize}\nn_iter: {n_iter} \ninit_points: {init_points} \n{settings}\n\n{result}'
    print(doc_string)

    path_prefix = PROJECT_ROOT / 'data/hyper_optimization_runs/'
    path_prefix.mkdir(parents=True, exist_ok=True)
    now_short = now.strftime('%Y%m%d%H%M%S')
    file_name = f'hyper_optim_run_{now_short}.txt'

    doc_string = doc_string + '\n ---------------------------------\n'
    for el in optimizer.res:
        doc_string = f'{doc_string}\n{el}'

    with open(path_prefix / file_name, 'w') as f:
        f.write(doc_string)

run_optimizer(test_max_model, pbounds)

2023-12-10 00:23:08

f: <function test_max_model at 0x7f200d6d3ec0>
n_iter: 10 
init_points: 10 
decay: !!python/tuple
- 0.0001
- 0.1
hidden_size: !!python/tuple
- 2
- 50.5
learning_rate: !!python/tuple
- 0.0001
- 0.01
n_hidden: !!python/tuple
- 1
- 3.5


params:
  decay: 0.062376230059230896
  hidden_size: 12.826390025166676
  learning_rate: 0.009595037005529505
  n_hidden: 3.011712537425001
target: -0.2543922073461793

